# ART SLIDE — モナ・リザのアイコン生成
原画を切り出し、5×5の合法手のみでシャッフルする。右目と口のピースを指定位置に配置し、逆順で解けることを検証する。

In [ ]:
from PIL import Image, ImageOps, ImageDraw, ImageFilter
import io, requests, random, math, json
from IPython.display import display
from google.colab import files

# 原画は Wikimedia Commons のパブリックドメイン作品。取得失敗時は手元の画像をアップロード。
URL = 'https://upload.wikimedia.org/wikipedia/commons/6/6a/Mona_Lisa.jpg'
try:
    r = requests.get(URL, headers={'User-Agent':'ArtSlideNotebook/1.0 (educational image puzzle)'}, timeout=30)
    r.raise_for_status()
    original = Image.open(io.BytesIO(r.content)).convert('RGB')
except Exception as e:
    print('原画をアップロードしてください:', e)
    uploaded = files.upload()
    original = Image.open(io.BytesIO(next(iter(uploaded.values())))).convert('RGB')
display(original.copy().resize((320, round(320*original.height/original.width))))

In [ ]:
# 原画に対する相対座標 (左, 上, 右, 下)。正方形に切り出す。
# 必要なら数値を調整。右目・口の元ピース番号は次のプレビューで確認。
SIZE = 1200
N = 5
T = SIZE // N
w,h = original.size
side = min(w*0.77,h*0.58)
cx,cy = w*0.50,h*0.29
x = max(0,min(w-side,cx-side/2))
y = max(0,min(h-side,cy-side/2))
art = original.crop((round(x),round(y),round(x+side),round(y+side))).resize((SIZE,SIZE),Image.Resampling.LANCZOS)
tiles = [art.crop((c*T,r*T,(c+1)*T,(r+1)*T)) for r in range(N) for c in range(N)]
preview = art.copy()
d = ImageDraw.Draw(preview)
for i in range(25):
    r,c=divmod(i,N)
    d.rectangle((c*T,r*T,(c+1)*T-1,(r+1)*T-1),outline='white',width=3)
    d.rectangle((c*T+5,r*T+5,c*T+44,r*T+30),fill='black')
    d.text((c*T+10,r*T+9),str(i+1),fill='white')
display(preview.resize((600,600)))

In [ ]:
# プレビューから右目・口が入っている番号を選ぶ（1始まり）。
# 既定値は目安。異なる場合はここを変更。
RIGHT_EYE = 14
MOUTH = 18
TARGETS = {RIGHT_EYE-1: 13, MOUTH-1: 17}
BLANK = 24
assert len(TARGETS)==2 and BLANK not in TARGETS
# 完成状態から合法手を重ね、条件を満たす候補だけ採用。
def neighbors(p):
    r,c=divmod(p,5)
    return [rr*5+cc for rr,cc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)] if 0<=rr<5 and 0<=cc<5]

def generate(seed=42, count=5, iterations=350000):
    rng=random.Random(seed)
    board=list(range(25)); blank=24; prev=-1
    history=[]; found=[]; seen=set()
    # 定期的に完成状態へ戻すことで、短い解法も得る。
    for step in range(iterations):
        if step%240==0:
            board=list(range(25)); blank=24; prev=-1; history=[]
        nxt=rng.choice([p for p in neighbors(blank) if p!=prev])
        board[blank],board[nxt]=board[nxt],board[blank]
        history.append((blank,nxt)); prev,blank=blank,nxt
        if len(history)<15 or not all(board[pos]==tile for tile,pos in TARGETS.items()):
            continue
        if blank not in [4,9,14,19,21,22,23,24]: continue
        if sum(a!=b for a,b in enumerate(board))<8: continue
        # 空白の上が明るめのピースになるよう優先
        if blank>=5:
            above=tiles[board[blank-5]].convert('L').resize((1,1)).getpixel((0,0))
            if above<95: continue
        key=tuple(board)
        if key in seen: continue
        if any(sum(a!=b for a,b in zip(board,old))<7 for old,_ in found): continue
        seen.add(key); found.append((board.copy(),history.copy()))
        if len(found)==count: break
    if len(found)<count: raise RuntimeError('候補不足。iterations を増やすか、ピース番号を確認してください。')
    return found

def verify(board, history):
    b=board.copy()
    assert sorted(b)==list(range(25))
    for a,z in reversed(history):
        assert b[z]==BLANK and a in neighbors(z)
        b[a],b[z]=b[z],b[a]
    assert b==list(range(25))
    return True

variants=generate()
for board,history in variants: assert verify(board,history)
print('全候補の復元を検証済み')

In [ ]:
# ピースの絵柄は変更せず、縁の陰影だけを追加する。
from PIL import ImageEnhance
def render(board, gap=4, radius=10, outer_radius=240):
    out=Image.new('RGBA',(SIZE,SIZE),(59,39,25,255))
    for p,tile in enumerate(board):
        if tile==BLANK: continue
        r,c=divmod(p,5)
        piece=tiles[tile].resize((T-gap,T-gap),Image.Resampling.LANCZOS).convert('RGBA')
        mask=Image.new('L',piece.size)
        ImageDraw.Draw(mask).rounded_rectangle((0,0,piece.width-1,piece.height-1),radius,fill=255)
        overlay=Image.new('RGBA',piece.size)
        d=ImageDraw.Draw(overlay)
        for k in range(5):
            d.rounded_rectangle((k,k,piece.width-1-k,piece.height-1-k),max(1,radius-k),outline=(255,220,170, max(0,65-k*12)) if k<2 else (30,14,5, max(0,45-k*7)),width=1)
        piece=Image.alpha_composite(piece,overlay)
        out.paste(piece,(c*T+gap//2,r*T+gap//2),mask)
    mask=Image.new('L',out.size)
    ImageDraw.Draw(mask).rounded_rectangle((0,0,SIZE-1,SIZE-1),outer_radius,fill=255)
    out.putalpha(mask)
    return out
display(render(list(range(25))).resize((400,400)))
for i,(board,history) in enumerate(variants,1):
    icon=render(board)
    icon.save(f'art_slide_icon_{i}.png')
    display(icon.resize((400,400)))
with open('puzzle_layouts.json','w') as f:
    json.dump([{'board':[x+1 if x!=BLANK else 0 for x in b], 'reverse_moves':list(reversed(h))} for b,h in variants],f,indent=2)
import zipfile
with zipfile.ZipFile('art_slide_icons.zip','w') as z:
    for i in range(1,len(variants)+1): z.write(f'art_slide_icon_{i}.png')
    z.write('puzzle_layouts.json')
files.download('art_slide_icons.zip')